In [ ]:
import sqlite3
from pathlib import Path

from crossfiledialog import choose_folder

from src.tag.yolo import YOLOTagger

&nbsp;

### **📂 Open a collection**

In [ ]:
collection_path = Path(choose_folder())
print(f"Selected collection: {collection_path}")

&nbsp;

### **🍵 Load tagger and database**

In [ ]:
tagger = YOLOTagger()
index_db = sqlite3.connect(collection_path / "index.db")
index_db.row_factory = sqlite3.Row

&nbsp;

### **🔍 Select ids and paths**

In [ ]:
images = index_db.execute("SELECT id, hash, fullname, tags FROM images").fetchall()
images = [{
    "id": img["id"],
    "path": collection_path / "images" / img["hash"] / img["fullname"],
    "tags": img["tags"],
} for img in images]
print(f"Selected {len(images)} images")
images[:5]

&nbsp;

### **🏷️ Tag it!**

In [ ]:
from IPython.display import Image, display

for i, image in enumerate(images):
    tags = tagger.tag(image["path"])
    image["tags"] = "|".join(list(set(image["tags"].split("|") + tags))).strip("|")
    
    if i < 5:
        display(Image(filename=image["path"].parent / "thumbnail.png", width=96))
        print(f"Tags: {tags}")

&nbsp;

### ✍️ **Write back to database (append)**

In [ ]:
for image in images:
    index_db.execute("UPDATE images SET tags = ? WHERE id = ?", (image["tags"], image["id"]))
index_db.commit()